# Transformación del dataset (audio) en ventanas de n segundos con label

### Instalar e importar librerías necesarias

In [ ]:
!pip install selenium
!pip install pdfrw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.2/499.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 1.4 MB/s eta 0:00:00
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 kB 4.1 MB/s eta 0:00:00


In [ ]:
!pip install kaggle

In [ ]:
import librosa
import IPython.display as ipd
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Audio
import soundfile as sf
from numpy import pi
from scipy.fftpack import fft, fftfreq

### Importar dataset (cuando no está)



In [ ]:
from google.colab import files

!chmod 600 ~/.kaggle/kaggle.json

In [35]:
import os
import json

if not os.path.exists('/root/.kaggle'):
    os.makedirs('/root/.kaggle')

!mv kaggle.json /root/.kaggle/

!chmod 600 /root/.kaggle/kaggle.json

print("Kaggle API key setup complete.")

mv: cannot stat 'kaggle.json': No such file or directory
Kaggle API key setup complete.


In [ ]:
# Descargar el dataset. Reemplaza 'israaelmorsy/lung-sound-classification' con el nombre de usuario/dataset correcto si es diferente.
!kaggle datasets download -d vbookshelf/respiratory-sound-database

Dataset URL: https://www.kaggle.com/datasets/vbookshelf/respiratory-sound-database
License(s): unknown
100% 3.69G/3.69G [00:44<00:00, 291MB/s]
100% 3.69G/3.69G [00:44<00:00, 89.7MB/s]


In [ ]:
!unzip respiratory-sound-database.zip

Archive:  respiratory-sound-database.zip
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Al_sc_Meditron.txt  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Al_sc_Meditron.wav  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Pr_sc_Meditron.txt  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Pr_sc_Meditron.wav  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/102_1b1_Ar_sc_Meditron.txt  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/102_1b1_Ar_sc_Meditron.wav  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/103_2b2_Ar_mc_LittC2SE.txt  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/103_2b2_Ar_mc_LittC2SE.wav  
  inflating: Respiratory_Sound_

### Obtener los audios

In [ ]:
import os
import librosa

audio_dir = '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/'
audio_data_list = []

for filename in os.listdir(audio_dir):
    if filename.endswith('.wav'):
        file_path = os.path.join(audio_dir, filename)
        y, sr = librosa.load(file_path, sr=None) # Cargar los audios con librosa
        audio_data_list.append(y)

print(f"Carga de {len(audio_data_list)} archivos de audio.")

Carga de 920 archivos de audio.


In [ ]:
max_len = max(len(audio) for audio in audio_data_list)
preprocessed_audio_data = []

for audio in audio_data_list:
    if len(audio) < max_len:
        padding = np.zeros(max_len - len(audio)) # Añade ceros
        padded_audio = np.concatenate((audio, padding))
        preprocessed_audio_data.append(padded_audio)
    elif len(audio) > max_len:
        truncated_audio = audio[:max_len]
        preprocessed_audio_data.append(truncated_audio)
    else:
        preprocessed_audio_data.append(audio)

print(f"Número de muestras: {len(audio_data_list)}")
print(f"Audios preprocesados: {len(preprocessed_audio_data)}")

Número de muestras: 920
Audios preprocesados: 920


In [ ]:
# Leer un audio para hacerle las pruebas

audio_file_path = None
for filename in os.listdir(audio_dir):
    if filename.endswith('.wav'):
        audio_file_path = os.path.join(audio_dir, filename)
        break

if audio_file_path:
    y, sr = librosa.load(audio_file_path) # Se define el audio como una señal de onda y sr referente al sample rate
    print(f"Successfully loaded audio file: {audio_file_path}")
else:
    print("No .wav files found in the directory.")

Successfully loaded audio file: /content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/207_3b2_Pr_mc_AKGC417L.wav


In [ ]:
Audio(y, rate=sr)
print(Audio(y,rate=sr))

<IPython.lib.display.Audio object>


In [ ]:
#Estudie cada uno de los parámetros del objeto creado para leer el archivo de audio.
F= sr #sample rate o frecuencia del audio
dt = y.size / sr  #
T = 1 / sr #Periodo
t = np.r_[0:dt:T]
print(
    f'y[t] tiene {y.size} muestras', #Cantidad de muestras en el audio
    f'la frecuencia de muestreo es {sr} Hz',
    f'y(t) tiene {dt:.1f}s '#duracion
    , sep='\n')


y[t] tiene 441000 muestras
la frecuencia de muestreo es 22050 Hz
y(t) tiene 20.0s 


In [29]:
# Calcular el número de muestras por ventana de n segundos

n = 4  # Duración de la ventana en segundos (le puse 4)
samples_per_window = n * sr

# Crear ventanas de 4 segundos
audio_windows = []
for i in range(0, len(y), samples_per_window):
    window = y[i:i + samples_per_window]
    audio_windows.append(window)

print(f"El audio se ha dividido en {len(audio_windows)} ventanas de aproximadamente 4 segundos cada una.")

El audio se ha dividido en 5 ventanas de aproximadamente 4 segundos cada una.


In [32]:
# Escuchar el audio de 4 segundos (Ventana 1) original

print("Ventana 1:")
display(Audio(audio_windows[0], rate=sr))

Ventana 1:


In [ ]:
from scipy.signal import butter, filtfilt

# Definir los parámetros
lowcut = 45.0  # Lower cutoff (Hz)
highcut = 55.0 # Upper cutoff (Hz)
order = 5      # Orden del filtro (no estoy seguro de cuánto debería ser)

# Implementación del filtro
nyquist = 0.5 * sr
low = lowcut / nyquist
high = highcut / nyquist
b, a = butter(order, [low, high], btype='band')

# Aplicar el filtro Butterworth
filtered_y = filtfilt(b, a, y)

print("Filtro pasabanda aplicado.")

Filtro pasabanda aplicado.


In [28]:
# Calcular el número de muestras por ventana de n segundos

n = 4  # Duración de la ventana en segundos (le puse 4)
samples_per_window = n * sr

# Crear ventanas de 4 segundos a partir del audio filtrado
audio_windowsF = []
for i in range(0, len(filtered_y), samples_per_window):
    windowF = filtered_y[i:i + samples_per_window]
    audio_windowsF.append(windowF)

print(f"El audio filtrado se ha dividido en {len(audio_windows)} ventanas de aproximadamente 4 segundos cada una.")

El audio filtrado se ha dividido en 5 ventanas de aproximadamente 4 segundos cada una.


In [31]:
# Escuchar el audio de 4 segundos (Ventana 1) filtrado

print("Ventana 1:")
display(Audio(audio_windowsF[0], rate=sr))

Ventana 1:


/usr/local/lib/python3.12/dist-packages/IPython/lib/display.py:175: RuntimeWarning: invalid value encountered in cast
  return scaled.astype("<h").tobytes(), nchan


In [34]:
# Descargar

import soundfile as sf
from google.colab import files

output_filename_filtered = 'primera_ventana_filtrada.wav'
sf.write(output_filename_filtered, audio_windowsF[0], sr)

files.download(output_filename_filtered)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Crear una lista de tuplas (ventana de audio, label)
labeled_audio_windows = []
for i, window in enumerate(audio_windows):
    label = f"ventana_{i+1}"  # Label de prueba, toca cambiarlo
    labeled_audio_windows.append((window, label))

print(f"Se han creado {len(labeled_audio_windows)} ventanas de audio con labels.")
print(labeled_audio_windows[0])

Se han creado 5 ventanas de audio con labels.
(array([0.        , 0.        , 0.        , ..., 0.02039468, 0.02351628,
       0.02455851], dtype=float32), 'ventana_1')


In [38]:
# Esta conversión a tensor me toca revisarla mejor

import torch
import numpy as np

# Separar los datos de audio y las etiquetas

audio_data_list = [item[0] for item in labeled_audio_windows]
labels = [item[1] for item in labeled_audio_windows]

# Convertir la lista de arrays de numpy a una lista de tensores de torch
audio_tensors = [torch.from_numpy(audio).float() for audio in audio_data_list]

# Apilar los tensores para crear un único tensor, asumiendo que todos los tensores tienen el mismo tamaño después del padding
try:
    audio_tensor = torch.stack(audio_tensors)
    print("Datos de audio convertidos a tensor.")
    print(f"Forma del tensor de audio: {audio_tensor.shape}")
except RuntimeError as e:
    print(f"Error al apilar tensores: {e}")
    print("Esto puede ocurrir si las ventanas de audio tienen tamaños diferentes.")
    print("Considera rellenar o truncar a una longitud uniforme antes de apilar.")

# Convertir etiquetas a un tensor

unique_labels = list(set(labels))
label_mapping = {label: i for i, label in enumerate(unique_labels)}
encoded_labels = [label_mapping[label] for label in labels]
label_tensor = torch.tensor(encoded_labels)

print(f"Etiquetas (primeras 5): {labels[:5]}")
print(f"Etiquetas codificadas (primeras 5): {encoded_labels[:5]}")
print(f"Forma del tensor de etiquetas: {label_tensor.shape}")

Datos de audio convertidos a tensor.
Forma del tensor de audio: torch.Size([5, 88200])
Etiquetas (primeras 5): ['ventana_1', 'ventana_2', 'ventana_3', 'ventana_4', 'ventana_5']
Etiquetas codificadas (primeras 5): [3, 1, 0, 4, 2]
Forma del tensor de etiquetas: torch.Size([5])


### Lectura de .txt para el síntoma

In [ ]:
txt_dir = '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/'
txt_files_list = []

for filename in os.listdir(txt_dir):
    if filename.endswith('.txt'):
        file_path = os.path.join(txt_dir, filename)
        txt_files_list.append(file_path)

print(f"Se hallaron {len(txt_files_list)} archivos .txt.")

Se hallaron 920 archivos .txt.


In [ ]:
# Obtener el nombre del audio que se escogió arriba
audio_basename = os.path.splitext(os.path.basename(audio_file_path))[0]

# Hallar el .txt con el mismo nombre
corresponding_txt_file = None
for txt_file_path in txt_files_list:
    txt_basename = os.path.splitext(os.path.basename(txt_file_path))[0]
    if txt_basename == audio_basename:
        corresponding_txt_file = txt_file_path
        break

# Leer el .txt y obtener la información de las dos últimas columnas, el síntoma
if corresponding_txt_file:
    print(f"Últimas dos columnas del .txt ({os.path.basename(corresponding_txt_file)}):")
    with open(corresponding_txt_file, 'r') as f:
        for line in f:
            columns = line.strip().split()
            if len(columns) >= 2: # Verificar que si eran dos columnas
                print(columns[-2:])

Últimas dos columnas del .txt (207_3b2_Pr_mc_AKGC417L.txt):
['0', '0']
['0', '0']
['0', '0']
['0', '0']
['0', '0']
['0', '0']
['0', '0']
['0', '0']
['0', '0']
